In [31]:
%pip install "ray[default]"
%pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 23.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [32]:
import os 
import ray 

In [33]:
import sys; sys.path.append("..")
import warnings; warnings.filterwarnings("ignore")
from dotenv import load_dotenv; load_dotenv()
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [34]:
if ray.is_initialized():
    ray.shutdown()
ray.init()

2026-06-15 22:39:50,459	INFO worker.py:2003 -- Started a local Ray instance. View the dashboard at 127.0.0.1:8265 


Python version:,3.10.11
Ray version:,2.55.1
Dashboard:,http://127.0.0.1:8265


In [35]:
ray.cluster_resources()

{'object_store_memory': 1316841062.0,
 'GPU': 1.0,
 'memory': 3072629146.0,
 'accelerator_type:G': 1.0,
 'node:127.0.0.1': 1.0,
 'node:__internal_head__': 1.0,
 'CPU': 16.0}

In [36]:
num_workers = 3
resources_per_worker={"CPU": 4, "GPU": 0.25}

In [37]:
import os
from pathlib import Path

if os.path.exists("/efs"):
    EFS_DIR = f"/efs/shared_storage/MY-FIRST-MLOPS-PROJECT/{os.environ.get('Eva', 'default_user')}"
else:    
    EFS_DIR = str(Path(os.getcwd()).parent / "local_storage")
    os.makedirs(EFS_DIR, exist_ok=True)

print(f"Current active storage directory: {EFS_DIR}")

Current active storage directory: /efs/shared_storage/MY-FIRST-MLOPS-PROJECT/default_user


In [38]:
import pandas as pd 

In [39]:
dataset = "https://raw.githubusercontent.com/evasim/my-first-MLOPS-project/refs/heads/main/data/raw_dataset.csv"
df = pd.read_csv(dataset)
df.head()

,text,label
0,Wall St. Bears Claw Back Into the Black (Reute...,2
1,Carlyle Looks Toward Commercial Aerospace (Reu...,2
2,Oil and Economy Cloud Stocks' Outlook (Reuters...,2
3,Iraq Halts Oil Exports from Main Southern Pipe...,2
4,"Oil prices soar to all-time record, posing new...",2


In [40]:
from sklearn.model_selection import train_test_split

In [41]:
df.label.value_counts()

label
2    30000
3    30000
1    30000
0    30000
Name: count, dtype: int64

In [42]:
test_size = 0.2 
train_df, val_df = train_test_split(df, stratify=df.label, test_size=test_size, random_state=42)

In [43]:
train_df.label.value_counts()

label
1    24000
3    24000
0    24000
2    24000
Name: count, dtype: int64

In [44]:
val_df.label.value_counts() * int((1 - test_size) / test_size)

label
2    24000
1    24000
0    24000
3    24000
Name: count, dtype: int64

In [45]:
from collections import Counter 
import matplotlib.pyplot as plt 
import seaborn as sns; sns.set_theme()
import warnings; warnings.filterwarnings("ignore")
from wordcloud import WordCloud, STOPWORDS

In [46]:
tags = Counter(df.label)
tags.most_common()

[(2, 30000), (3, 30000), (1, 30000), (0, 30000)]

In [47]:
import json 
import nltk 
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
import re

In [48]:
nltk.download("stopwords")
words = stopwords.words("english")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [49]:
df.head()

,text,label
0,Wall St. Bears Claw Back Into the Black (Reute...,2
1,Carlyle Looks Toward Commercial Aerospace (Reu...,2
2,Oil and Economy Cloud Stocks' Outlook (Reuters...,2
3,Iraq Halts Oil Exports from Main Southern Pipe...,2
4,"Oil prices soar to all-time record, posing new...",2


In [50]:
def clean_text(text, stopwords=words):
    # change every words into lower case
    text = text.lower()

    # removing stopwords such as "is", "the" and so on
    pattern = re.compile(r'\b(' + r"|".join(words)+ r")\b\s*")
    text = pattern.sub('', text)

    text = re.sub(r"([!\"'#$%&()*\+,-./:;<=>?@\\\[\]^_`{|}~])", r" \1 ", text) # add space 
    text = re.sub("[^A-Za-z0-9]+", " ", text) # remove other than words and numbers 
    text = re.sub(" +", " ", text) # remove all extra spaces 
    text = text.strip() # remove spaces at the start and at the end 
    text = re.sub(r"http\S+", "", text) # remove links 

    return text

In [51]:
ori_df = df.copy()
df.text = df.text.apply(clean_text)
print(f"{ori_df.text.values[0]}\n{df.text.values[0]}")

Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\band of ultra-cynics, are seeing green again.
wall st bears claw back black reuters reuters short sellers wall street dwindling band ultra cynics seeing green


In [52]:
df = df.dropna(subset=["label"]) 
df.head()

,text,label
0,wall st bears claw back black reuters reuters ...,2
1,carlyle looks toward commercial aerospace reut...,2
2,oil economy cloud stocks outlook reuters reute...,2
3,iraq halts oil exports main southern pipeline ...,2
4,oil prices soar time record posing new menace ...,2


In [53]:
label_decoder = {
    0: "World",
    1: "Sports",
    2: "Business",
    3: "Sci/Tech"
}

In [54]:
import numpy as np 
from transformers import BertTokenizer

In [55]:
def tokenize(batch):
    tokenizer = BertTokenizer.from_pretrained("allenai/scibert_scivocab_uncased", return_dict= False)
    encoded = tokenizer(batch["text"].tolist(), return_tensors="np", padding = "longest")
    return dict(ids=encoded["input_ids"], masks=encoded["attention_mask"], targets=np.array(batch["label"]))

In [56]:
tokenize(df.head(1))

{'ids': array([[  102,  3545,   177, 23309,  3895, 30128,  1542,  3778,   144,
         13342, 30113,   144, 13342, 30113,  2001, 27316,  3545, 10833,
         10029,   484,  2123,  2102, 10186, 27637,  1081, 18934,  3755,
           103]]),
 'masks': array([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1]]),
 'targets': array([2])}

In [57]:
# combining all preprocessing steps into function
def preprocess(df):
    df["text"] = df.text.apply(clean_text)
    targets = tokenize(df)
    return targets 

In [58]:
preprocess(df=train_df)

{'ids': array([[  102,   326,  2283, ...,     0,     0,     0],
        [  102, 19723, 15953, ...,     0,     0,     0],
        [  102,  3293,  1482, ...,     0,     0,     0],
        ...,
        [  102,   253, 17553, ...,     0,     0,     0],
        [  102, 23314, 30113, ...,     0,     0,     0],
        [  102,  3841,  2579, ...,     0,     0,     0]],
       shape=(96000, 199)),
 'masks': array([[1, 1, 1, ..., 0, 0, 0],
        [1, 1, 1, ..., 0, 0, 0],
        [1, 1, 1, ..., 0, 0, 0],
        ...,
        [1, 1, 1, ..., 0, 0, 0],
        [1, 1, 1, ..., 0, 0, 0],
        [1, 1, 1, ..., 0, 0, 0]], shape=(96000, 199)),
 'targets': array([1, 3, 0, ..., 3, 2, 3], shape=(96000,))}